# `c08_om` — Outcome Measures for Non-Traditional Cohorts

**Component curation notebook.** Fetches the raw IPEDS distribution files, verifies the
reference period against official documentation, locks the schema, reshapes to the
declared grain, validates, and writes one curated table with a metadata sidecar.

| Property | Value |
|---|---|
| Native tables | `OM2023`, `DRVOM2023` |
| Reference period | 2015-16 entering cohort observed at 4, 6, and 8 years (August 31 2019, 2021, and 2023) |
| Curated grain | `UNITID` x `OMCHRT` |
| Output | `data/curated/c08_om.parquet` |

Outcome Measures exists because the Graduation Rates cohort excludes part-time and non-first-time students, which at open-access and community institutions is most of the student body. OMCHRT splits the cohort by entry status, and those splits are the point of the component rather than a nuisance dimension.

> **Pitfall.** OMENRUN is measured ignorance, not zero. It counts students whose subsequent enrollment status could not be determined. Treating it as a non-completion inflates apparent failure at institutions with poor match rates to the National Student Clearinghouse. Carry OMENRUP as an explicit uncertainty band: the honest completion estimate is an interval whose width is that unknown share, not a point.

## 1. Environment

One import surface, so a parsing quirk is fixed once rather than twelve times.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import ipeds_utils as iu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

SLUG = "c08_om"
TABLES = ['OM2023', 'DRVOM2023']
GRAIN = ['UNITID', 'OMCHRT']
REFERENCE_PERIOD = '2015-16 entering cohort observed at 4, 6, and 8 years (August 31 2019, 2021, and 2023)'

print("ipeds_utils", iu.__version__, "| pandas", pd.__version__)

ipeds_utils 1.1.0 | pandas 3.0.5


## 2. Retrieve

Downloads are cached, so re-running this notebook is offline and cheap. Every retrieval returns a provenance record carrying a SHA-256 digest, which is what makes a result reproducible rather than merely repeatable.

In [2]:
RAW_DIR = "../data/raw"   # relative to notebooks/, so all twelve share one cache

provenance = [iu.fetch(t, raw_dir=RAW_DIR) for t in TABLES]
pd.DataFrame(provenance)[["table", "data_bytes", "data_sha256", "retrieved_utc"]]

,table,data_bytes,data_sha256,retrieved_utc
0,OM2023,1347816,60a619776e8da60542c9a728b2c2122bcdef6825c76b5b...,2026-09-24T17:19:28+00:00
1,DRVOM2023,260697,edf3a1d13d7e126f9413fbad0bf79d9ba320abad8d5f52...,2026-09-24T17:19:28+00:00


## 3. Verify the reference period

**Do not skip this cell.** The filename year is not the reference period, and the offsets are not uniform across components. This assertion fails loudly rather than letting a misaligned period corrupt every downstream year comparison, where it would be invisible in the data itself.

In [3]:
intro = iu.assert_reference_period(
    provenance[0]["dict_path"],
    expect=r'2015-16',
    table=TABLES[0],
)
print(intro[:600])

File documentation for Outcome Measures of entering degree/certificate-seeking undergraduate cohorts from 2015-16 at degree-granting institutions, by Pell status:  August 31, 2023
(Provisional release)
Filename OM2023
Overview This table contains award and enrollment data from degree-granting institutions on four cohorts and eight subcohorts of undergraduates who entered an institution in 2015-16 at three points in time: four-year (August 31, 2019) six-year (August 31, 2021) and eight-year (August 31, 2023).  The cohorts of degree/certificate-seeking undergraduates are:

(1) First-time full-ti


## 4. Inspect the dictionary

Variable labels come from the published dictionary, never from memory. This is also where value sets are read, so categorical decoding is driven by the official codebook and a taxonomy revision surfaces as unmatched codes instead of a plausible-looking wrong label.

In [4]:
variables = iu.read_dict(provenance[0]["dict_path"])
valuesets = iu.read_valuesets(provenance[0]["dict_path"])

print(f"{len(variables)} variables documented, {len(valuesets)} value-set rows")
variables[["varname", "vartitle"]].head(20)

28 variables documented, 15 value-set rows


,varname,vartitle
0,UNITID,Unique identification number of the institution
1,OMCHRT,Cohort category
2,OMRCHRT,2015-16 cohort
3,OMEXCLS,Exclusions to 2015-16 cohort
4,OMACHRT,Adjusted 2015-16 cohort
5,OMCERT4,Number of adjusted cohort receiving a certific...
6,OMASSC4,Number of adjusted cohort receiving an Associa...
7,OMBACH4,Number of adjusted cohort receiving a Bachelor...
8,OMAWDN4,Number of adjusted cohort receiving an award a...
9,OMAWDP4,Percent of adjusted cohort receiving an award ...


## 5. Load and lock the schema

The first run records the column signature; later runs fail if it drifts.

In [5]:
KEEP = ['UNITID', 'OMCHRT', 'OMRCHRT', 'OMACHRT', 'OMAWDN4', 'OMAWDN6', 'OMAWDN8', 'OMAWDP8', 'OMENRYI', 'OMENRAI', 'OMENRUN', 'OMNOAWD', 'OMENRUP', 'OMENRTP']

raw = iu.read_csv(provenance[0]["data_path"])
print("raw shape", raw.shape)

lock = iu.lock_schema(raw, TABLES[0], schema_dir="../schemas", strict=False)
print("schema:", lock["status"], "| added", lock["added"][:5], "| removed", lock["removed"][:5])

available = [c for c in KEEP if c in raw.columns]
missing = [c for c in KEEP if c not in raw.columns]
if missing:
    print("NOT PRESENT in this cycle (verify against the varlist above):", missing)

frame = raw[available].copy()
frame.head()

raw shape (47342, 54)
schema: unchanged | added [] | removed []


,UNITID,OMCHRT,OMRCHRT,OMACHRT,OMAWDN4,OMAWDN6,OMAWDN8,OMAWDP8,OMENRYI,OMENRAI,OMENRUN,OMNOAWD,OMENRUP,OMENRTP
0,100654,10,1374,1373,143,369,399,29.0,9,492,473,974,34.0,36.0
1,100654,11,1037,1036,94,279,301,29.0,8,350,377,735,36.0,35.0
2,100654,12,337,337,49,90,98,29.0,1,142,96,239,28.0,42.0
3,100654,20,189,188,1,10,12,6.0,2,69,105,176,56.0,38.0
4,100654,21,129,129,0,8,9,7.0,1,52,67,120,52.0,41.0


## 6. Mask reserved missing codes

IPEDS encodes missingness as negative integers. A mean computed without masking them is badly wrong and looks entirely plausible, which is what makes this the most costly single omission in IPEDS analysis.

In [6]:
RESERVED = [-1, -2, -3, -9]

numeric_cols = [
    c for c in frame.columns
    if c not in ("UNITID", *GRAIN) and pd.api.types.is_numeric_dtype(frame[c])
]

before = frame[numeric_cols].isna().sum().sum()
for col in numeric_cols:
    frame.loc[frame[col].isin(RESERVED), col] = np.nan
after = frame[numeric_cols].isna().sum().sum()

# Masking turns an integer column into float (1 becomes 1.0). Measures can stay float,
# since NaN is what the models expect, but category codes go back to nullable integers
# so they print, join, and decode as codes rather than as 1.0.
for col in ['OMCHRT']:
    if col in frame.columns and pd.api.types.is_float_dtype(frame[col]):
        if (frame[col].dropna() % 1 == 0).all():
            frame[col] = frame[col].astype("Int64")

print(f"masked {after - before:,} reserved-code cells across {len(numeric_cols)} numeric columns")

masked 0 reserved-code cells across 12 numeric columns


## 7. Carry the imputation flags

An imputed value and a reported value are not the same evidence. A column where most institutions carry a generated flag should not be modelled as though it were observed, and this is where that judgement becomes possible.

In [7]:
values, flags = iu.split_imputation_flags(raw, numeric_cols)

if flags.shape[1] > 1:
    summary = iu.imputation_summary(flags)
    display(summary.head(15))
    reported = summary[summary.flag == "R"].set_index("column")["share"]
    weak = reported[reported < 0.90]
    if len(weak):
        print("Columns under 90% reported — interpret with care:")
        display(weak)
else:
    print("No X-prefixed imputation flags accompany this file.")

,column,flag,n,share
1,XOMACHRT,R,47342,1.0000
2,XOMAWDN4,R,47321,0.9996
3,XOMAWDN4,P,21,0.0004
4,XOMAWDN6,R,47321,0.9996
5,XOMAWDN6,P,21,0.0004
6,XOMAWDN8,R,47321,0.9996
7,XOMAWDN8,P,21,0.0004
8,XOMAWDP8,R,47308,0.9993
9,XOMAWDP8,P,21,0.0004
10,XOMAWDP8,A,13,0.0003


## 8. Decode categoricals

Labels from the published value sets, not hand-typed mappings.

In [8]:
CATEGORICALS = ['OMCHRT']

unresolved = {}
for col in CATEGORICALS:
    if col in frame.columns:
        frame = iu.decode(frame, valuesets, col)
        unmatched = frame.loc[frame[col].notna() & frame[f"{col}_LABEL"].isna(), col].unique()
        if len(unmatched):
            unresolved[col] = sorted(unmatched.tolist())[:10]

# An unmatched code means a taxonomy change or a parsing fault. Either way the
# labels are wrong, so this stops the notebook rather than printing a warning.
assert not unresolved, f"codes absent from the published value set: {unresolved}"

label_cols = [c for c in frame.columns if c.endswith("_LABEL")]
frame[CATEGORICALS + label_cols].drop_duplicates().head(20) if label_cols else frame.head()

,OMCHRT,OMCHRT_LABEL
0,10,"First-time, full-time entering, Total"
1,11,"First-time, full-time entering, Pell Grant rec..."
2,12,"First-time, full-time entering, Non-Pell Grant..."
3,20,"First-time, part-time entering, Total"
4,21,"First-time, part-time entering, Pell Grant rec..."
5,22,"First-time, part-time entering, Non-Pell Grant..."
6,30,"Non-first-time, full-time entering, Total"
7,31,"Non-first-time, full-time entering, Pell Grant..."
8,32,"Non-first-time, full-time entering, Non-Pell G..."
9,40,"Non-first-time, part-time entering, Total"


## 9. Reshape to the declared grain

Target grain: `UNITID` x `OMCHRT`. The grain is asserted, not assumed, because a duplicated key silently inflates every aggregate computed downstream.

In [9]:
curated = frame.copy()

# This component already arrives at its declared grain, so curation is a
# pass-through. Components with a long layout (GRTYPE, EFFYALEV, STAFFCAT,
# OMCHRT) filter or pivot here instead; see c10_f for a full worked reshape.

present_grain = [g for g in GRAIN if g in curated.columns]
duplicated = curated.duplicated(subset=present_grain, keep=False).sum()
print(f"grain {present_grain} -> {len(curated):,} rows, {duplicated} duplicated")
assert duplicated == 0, "Declared grain is not unique; resolve before continuing."

curated.head()

grain ['UNITID', 'OMCHRT'] -> 47,342 rows, 0 duplicated


,UNITID,OMCHRT,OMRCHRT,OMACHRT,OMAWDN4,OMAWDN6,OMAWDN8,OMAWDP8,OMENRYI,OMENRAI,OMENRUN,OMNOAWD,OMENRUP,OMENRTP,OMCHRT_LABEL
0,100654,10,1374.0,1373.0,143.0,369.0,399.0,29.0,9.0,492.0,473.0,974.0,34.0,36.0,"First-time, full-time entering, Total"
1,100654,11,1037.0,1036.0,94.0,279.0,301.0,29.0,8.0,350.0,377.0,735.0,36.0,35.0,"First-time, full-time entering, Pell Grant rec..."
2,100654,12,337.0,337.0,49.0,90.0,98.0,29.0,1.0,142.0,96.0,239.0,28.0,42.0,"First-time, full-time entering, Non-Pell Grant..."
3,100654,20,189.0,188.0,1.0,10.0,12.0,6.0,2.0,69.0,105.0,176.0,56.0,38.0,"First-time, part-time entering, Total"
4,100654,21,129.0,129.0,0.0,8.0,9.0,7.0,1.0,52.0,67.0,120.0,52.0,41.0,"First-time, part-time entering, Pell Grant rec..."


## 10. Validate

Rules are declarative so the output is a persistable report: which checks ran, which failed, on how many rows, and which institutions were implicated. That report is the artefact you cite when claiming this table is fit for analysis.

In [10]:
RULES = [
    iu.unique_key('UNITID', 'OMCHRT'),
    iu.in_range('OMACHRT', 0, None),
    iu.Rule('adjusted_le_reported', lambda d: pd.to_numeric(d.OMACHRT, errors='coerce') > pd.to_numeric(d.OMRCHRT, errors='coerce'), note='Adjusted cohort cannot exceed the reported cohort'),
    iu.sums_to('OMACHRT', ['OMAWDN8', 'OMENRYI', 'OMENRAI', 'OMENRUN'], severity='warn'),
    iu.Rule('award_monotone_4_6_8', lambda d: (pd.to_numeric(d.OMAWDN8, errors='coerce') < pd.to_numeric(d.OMAWDN6, errors='coerce')) | (pd.to_numeric(d.OMAWDN6, errors='coerce') < pd.to_numeric(d.OMAWDN4, errors='coerce')), note='Cumulative awards cannot decrease as the window widens'),
]

report = iu.validate(curated, RULES, SLUG)
report.save(f"../reports/validation/{SLUG}.json")
display(report.to_frame()[["name", "status", "n_offending", "share", "note"]])

print("PASSED" if report.ok else "FAILED")
report.raise_if_failed()

,name,status,n_offending,share,note
0,"unique_key(UNITID,OMCHRT)",pass,0,0.0,Declared grain must be unique
1,"in_range(OMACHRT,0,None)",pass,0,0.0,Value plausibility bound
2,adjusted_le_reported,pass,0,0.0,Adjusted cohort cannot exceed the reported cohort
3,sums_to(OMACHRT),pass,0,0.0,Parts must reconcile within 0
4,award_monotone_4_6_8,pass,0,0.0,Cumulative awards cannot decrease as the windo...


PASSED


Report(table='c08_om', rows=47342, results=[{'name': 'unique_key(UNITID,OMCHRT)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Declared grain must be unique', 'status': 'pass'}, {'name': 'in_range(OMACHRT,0,None)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Value plausibility bound', 'status': 'pass'}, {'name': 'adjusted_le_reported', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Adjusted cohort cannot exceed the reported cohort', 'status': 'pass'}, {'name': 'sums_to(OMACHRT)', 'severity': 'warn', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Parts must reconcile within 0', 'status': 'pass'}, {'name': 'award_monotone_4_6_8', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Cumulative awards cannot decrease as the window widens', 'status': 'pass'}], generated_utc='2026-09-24T17:19:28+00:00')

## 11. Write the curated table

The sidecar carries the reference period and grain with the data. This is the defence against assembling a panel by filename year when the underlying periods are offset differently per component.

In [11]:
path = iu.write_curated(
    curated,
    SLUG,
    root="../data/curated",
    reference_period=REFERENCE_PERIOD,
    grain=GRAIN,
    provenance=provenance,
    notes='OMENRUN is measured ignorance, not zero. It counts students whose subsequent enrollment status could not be determined. Treating it as a non-completion inflates apparent failure at institutions with poor match rates to the National Student Clearinghouse. Carry OMENRUP as an explicit uncertainty band: the honest completion estimate is an interval whose width is that unknown share, not a point.',
)

iu.write_provenance(provenance, f"../docs/provenance/{SLUG}.json")
print("wrote", path, f"({len(curated):,} rows x {curated.shape[1]} columns)")

wrote ../data/curated/c08_om.parquet (47,342 rows x 15 columns)


## 12. Exercises

1. Re-run this notebook against the prior collection cycle by changing `TABLES`. The schema lock and the period assertion will both object; resolve each objection and record what changed between cycles.
2. Identify the three columns with the lowest reported-flag share, and argue whether each belongs in a predictive model at all.
3. Construct one derived cross-tabulation from this table, then apply `iu.suppress` and `iu.k_anonymity` to it. Report the smallest equivalence class before and after coarsening, and state the k you would require before publishing.
4. OMENRUN is measured ignorance, not zero. Write a validation rule that would catch this error if a colleague made it, and add it to `RULES` above.